# df DDPM sampling sweep

Compares DDIM step counts, eta values, and saved training epochs. The published baseline uses 50 steps and eta 0.0.

Recorded diagnostics found lower saturation (0.053 versus 0.188) and contrast (0.064 versus 0.145) in synthetic df than real df. This sweep measures whether those properties change with the sampler settings or checkpoint epoch.

Read-only against the fixed split: train and validation manifests only. Outputs are descriptive; the notebook does not select a model or retrain the generator.


In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "d2683ebe09b5124823fada3f39bc30ea201c55bb"
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the reviewed pushed commit before Run all"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
DIAGNOSTIC_VERSION = "v1_ddpm_sampling_sweep"

# Sweep grid. Widen only deliberately: every extra cell is another full sampling run.
SWEEP_STEPS = (50, 250, 1000)   # 50 reproduces the published set
SWEEP_ETAS = (0.0, 1.0)         # 0.0 reproduces the published set
EPOCH_SWEEP = (60, 80, 100)     # baseline setting across training progress
PRIMARY_EPOCH = 100
N_PER_CONFIG = 128              # ample for colour statistics; not a publication sample
BATCH_SIZE = 64
SEED = 0

SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
DDPM_CHECKPOINT_DIR = SHARED_PROJECT_DIR / "outputs" / "ddpm" / "checkpoints"
JUDGE_CHECKPOINT = SHARED_PROJECT_DIR / "outputs" / "classifier_df585" / "checkpoints" / "C1_seed2" / "best.pt"
DIAGNOSTIC_ROOT = SHARED_PROJECT_DIR / "outputs" / "diagnostics" / DIAGNOSTIC_VERSION
CODE_DIR = Path("/content/ddpm-code")

## Phase 0 — mount Drive, clone the pinned commit, install the generator dependency

In [ ]:
import json, os, subprocess, sys, time
from google.colab import drive
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project folder: {SHARED_PROJECT_DIR}"
assert DDPM_CHECKPOINT_DIR.is_dir(), f"missing DDPM checkpoints: {DDPM_CHECKPOINT_DIR}"
assert JUDGE_CHECKPOINT.is_file(), f"missing C1 judge checkpoint: {JUDGE_CHECKPOINT}"

assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.27", "pandas>=2.0", "pillow>=9.0"], check=True)

import torch
assert torch.cuda.is_available(), "a GPU runtime is required; Runtime -> Change runtime type -> GPU"
print(json.dumps({"commit": commit, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}, indent=2))

# The scripts resolve the fixed split through the repository's own config.
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
os.environ["DDPM_DERM_OUTPUTS_DIR"] = str(SHARED_PROJECT_DIR / "outputs")
env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONDONTWRITEBYTECODE"] = "1"

available = sorted(p.name for p in DDPM_CHECKPOINT_DIR.glob("run_seed0_epoch*.pt"))
print("checkpoints on Drive:", available)
for epoch in set(EPOCH_SWEEP) | {PRIMARY_EPOCH}:
    assert (DDPM_CHECKPOINT_DIR / f"run_seed0_epoch{epoch:04d}.pt").is_file(), f"missing epoch {epoch}"

## Phase 1: Streaming subprocess output


In [ ]:
import queue, threading

def run_stream(command, cwd=CODE_DIR, heartbeat=60):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, output_queue = [], queue.Queue()
    def pump():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=heartbeat)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic()-started:.0f}s alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
    return time.monotonic() - started, "".join(lines)

## Phase 2: Smoke run

Generates four images with five sampling steps to check imports, paths, and checkpoint compatibility. These measurements are not experimental results.


In [ ]:
smoke_dir = Path("/content/sweep_smoke")
elapsed, _ = run_stream([
    sys.executable, "-B", "-u", "scripts/ddpm_sampling_sweep.py",
    "--checkpoint-dir", str(DDPM_CHECKPOINT_DIR),
    "--judge-checkpoint", str(JUDGE_CHECKPOINT),
    "--steps", "5", "--etas", "0.0", "--epochs", str(PRIMARY_EPOCH),
    "--primary-epoch", str(PRIMARY_EPOCH),
    "--n", "4", "--batch-size", "4", "--seed", str(SEED),
    "--out", str(smoke_dir / "smoke.json"),
    "--montage", str(smoke_dir / "smoke.png"),
])
print(f"\n[smoke] COMPLETE in {elapsed:.0f}s -- the full sweep below is safe to start")

## Phase 3: Sampling sweep

Evaluates `len(SWEEP_STEPS) * len(SWEEP_ETAS)` settings at `PRIMARY_EPOCH`, then `EPOCH_SWEEP` at the published baseline setting. Runtime is dominated by the 1000-step settings. Completed outputs are saved to Drive.


In [ ]:
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
record_path = DIAGNOSTIC_ROOT / "sampling_sweep_record.json"
montage_path = DIAGNOSTIC_ROOT / "sampling_sweep_montage.png"
assert not record_path.exists(), f"a record already exists; move it aside deliberately: {record_path}"

elapsed, _ = run_stream([
    sys.executable, "-B", "-u", "scripts/ddpm_sampling_sweep.py",
    "--checkpoint-dir", str(DDPM_CHECKPOINT_DIR),
    "--judge-checkpoint", str(JUDGE_CHECKPOINT),
    "--steps", *[str(s) for s in SWEEP_STEPS],
    "--etas", *[str(e) for e in SWEEP_ETAS],
    "--epochs", *[str(e) for e in EPOCH_SWEEP],
    "--primary-epoch", str(PRIMARY_EPOCH),
    "--n", str(N_PER_CONFIG), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
    "--out", str(record_path), "--montage", str(montage_path),
])
print(f"\n[sweep] COMPLETE in {elapsed/60:.1f} min -> {record_path}")

## Phase 4: Diagnostic comparison

Compares saturation, contrast, and distances against real train df and the published `steps50_eta0.0` baseline. Changes across sampler settings or epochs describe sensitivity within the tested grid; they do not by themselves establish the cause of the image-quality gap.


In [ ]:
record = json.loads(record_path.read_text(encoding="utf-8"))
target = record["reference"]["real_train_df"]
print(f"target (real train df): saturation={target['saturation']:.4f}  contrast={target['contrast_std']:.4f}")
print(f"reference (real val df): embedding_nn_median={record['reference']['real_val_df']['embedding_nn_median']:.4f}\n")

header = f"{'configuration':22} {'saturation':>11} {'contrast':>9} {'embed_nn':>9} {'sat/real':>9}"
for bucket in ("sampler_sweep", "epoch_sweep"):
    print(f"--- {bucket} ---"); print(header)
    for tag, s in sorted(record[bucket].items()):
        print(f"{tag:22} {s['saturation']:11.4f} {s['contrast_std']:9.4f} "
              f"{s['embedding_nn_median']:9.4f} {s['saturation']/target['saturation']:9.2f}")
    print()

best = max(record["sampler_sweep"].items(), key=lambda kv: kv[1]["saturation"])
print(f"highest saturation: {best[0]} at {best[1]['saturation']:.4f} "
      f"({best[1]['saturation']/target['saturation']:.0%} of real df)")
print("published baseline steps50_eta0.0 is the control; it should match the recorded 0.053")

In [ ]:
from IPython.display import Image as ShowImage, display
print("rows:", ", ".join(record["montage"]["rows"]))
display(ShowImage(filename=str(montage_path)))